# User-based CF / Item-based CF 추천 모델 + 검증 데이터 구성

`docs/최종모델.md`의 유사도 기반 추천 설계와, 스팀 게임 추천 관련 선행연구인
「게임 분류와 MAB를 이용한 게임 추천 시스템」(이한우, 서강대 정보통신대학원, 2022) 논문의
협업 필터링(Collaborative Filtering) 배경 이론 및 평가 방법론을 참고하여 작성합니다.

이 노트북에서 하는 일:

1. `cleaned_data.csv`의 유저-게임 이용시간(`playtime_hours`)을 논문과 동일한 방식으로
   **유저별 min-max 정규화**하여 implicit rating으로 변환 (최소값은 유저별 최소값이 아닌 0으로 고정)
2. 유저별 게임 30개 중 **10%(3개)를 무작위로 마스킹**하여 검증 데이터(holdout)로 분리
3. 남은 90%(train)로 **User-based CF**와 **Item-based CF**(코사인 유사도 기반) 예측 모델 구현
4. 논문에서 사용한 평가지표인 **NDCG@10**과 Precision@10 / Recall@10으로 두 모델을 비교
5. 결과 해석 — 데이터 희소성(sparsity)이 성능에 미치는 영향 정리

> 참고: 논문은 스팀 API의 최근 2주 이용시간(`playtime_2weeks`)을 정규화 기준으로 사용했지만,
> 우리 데이터는 `recent_playtime_hours`가 대부분 0이라 신호로 쓰기 어려워 **총 이용시간(`playtime_hours`)**을
> 대신 사용합니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)

SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path('..') / 'data' / 'processed'
RESULTS_DIR = Path('..') / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

## 1. 데이터 로드 및 interaction 테이블 구성 (+ 유저 필터링)

`cleaned_data.csv`는 (유저, 게임) 조합 단위 row로 되어 있으므로, 추천에 필요한
`steamid`, `appid`, `game_name`, `playtime_hours`만 뽑아 interaction 테이블을 만듭니다.

팀에서 `clearingdataset` 브랜치(PR #3)로 데이터를 정제하면서, **30개 게임의 `playtime_hours`가
전부 0인 유저 99명**(플레이 기록이 전혀 없는 라이브러리 — 추천 신호가 없는 유저)을 제외했습니다.
`data/processed/user_profile_features_filtered.csv`가 이 정제 결과이므로, 여기 포함된 유저만
남기는 방식으로 CF 파이프라인에도 동일한 필터링 기준을 반영합니다.

In [2]:
df = pd.read_csv(DATA_DIR / 'cleaned_data.csv')
print(f"원본 shape: {df.shape[0]:,} rows / 유저 {df['steamid'].nunique():,}명 / 게임 {df['appid'].nunique():,}개")

# clearingdataset PR 기준 정제 유저만 사용 (playtime_hours 전부 0인 유저 99명 제외)
filtered_users = pd.read_csv(DATA_DIR / 'user_profile_features_filtered.csv')['steamid'].unique()
n_before = df['steamid'].nunique()
df = df[df['steamid'].isin(filtered_users)].reset_index(drop=True)
print(f"필터링 후: 유저 {df['steamid'].nunique():,}명 (제외 {n_before - df['steamid'].nunique():,}명) "
      f"/ 게임 {df['appid'].nunique():,}개")

game_names = df.drop_duplicates('appid').set_index('appid')['game_name']

inter = df[['steamid', 'appid', 'playtime_hours']].drop_duplicates(subset=['steamid', 'appid']).reset_index(drop=True)

games_per_user = inter.groupby('steamid').size()
print(f"유저별 게임 수: min={games_per_user.min()}, max={games_per_user.max()}, mean={games_per_user.mean():.1f}")

원본 shape: 26,340 rows / 유저 878명 / 게임 3,564개
필터링 후: 유저 779명 (제외 99명) / 게임 3,267개
유저별 게임 수: min=30, max=30, mean=30.0


## 2. Implicit Rating 생성

논문 4장 2절의 방식대로, 유저별 이용시간을 min-max normalization 하되 **최소값을 유저별 최소값이 아닌 0으로 고정**합니다.
그래야 가장 적게 플레이한 게임도 0이 아닌 값을 가져 선호 신호가 죽지 않습니다.

$$rating_{u,i} = \frac{playtime_{u,i}}{\max_i(playtime_{u,i})}$$

In [3]:
user_max = inter.groupby('steamid')['playtime_hours'].transform('max')
inter['rating'] = np.where(user_max > 0, inter['playtime_hours'] / user_max, 0.0)
inter[['steamid', 'appid', 'playtime_hours', 'rating']].head()

,steamid,appid,playtime_hours,rating
0,76561199478662488,105600,359.42,1.000000
1,76561199478662488,1281930,357.25,0.993962
2,76561199478662488,582010,121.65,0.338462
3,76561199478662488,1623730,120.80,0.336097
4,76561199478662488,504230,59.62,0.165878


## 3. Train / Validation Split — 유저별 게임 30개 중 10% 마스킹

유저마다 게임을 정확히 30개씩 보유하고 있으므로, 유저별로 무작위 3개(10%)를 검증용으로 떼어내고
나머지 27개(train)로 유사도/예측 모델을 학습합니다. 이렇게 하면 "실제로 플레이한 게임을 CF가
다시 찾아낼 수 있는가"를 검증할 수 있습니다 (leave-k-out 방식, 논문의 NDCG 평가와 동일한 발상).

In [4]:
HOLDOUT_FRAC = 0.1

is_holdout = np.zeros(len(inter), dtype=bool)
for uid, g in inter.groupby('steamid'):
    n = len(g)
    k = max(1, round(n * HOLDOUT_FRAC))
    chosen = rng.choice(g.index.values, size=k, replace=False)
    is_holdout[chosen] = True

train_df = inter[~is_holdout].reset_index(drop=True)
holdout_df = inter[is_holdout].reset_index(drop=True)

print(f"train: {train_df.shape[0]:,} / holdout: {holdout_df.shape[0]:,}")
print("유저별 holdout 게임 수:")
print(holdout_df.groupby('steamid').size().describe())

train_df.to_csv(DATA_DIR / 'cf_train_interactions.csv', index=False)
holdout_df.to_csv(DATA_DIR / 'cf_holdout_interactions.csv', index=False)
print("저장 완료: cf_train_interactions.csv, cf_holdout_interactions.csv")

train: 21,033 / holdout: 2,337
유저별 holdout 게임 수:
count    779.0
mean       3.0
std        0.0
min        3.0
25%        3.0
50%        3.0
75%        3.0
max        3.0
dtype: float64
저장 완료: cf_train_interactions.csv, cf_holdout_interactions.csv


## 4. User-Item 행렬 구성 (train 기준)

`train_df`만 사용해 유저 x 게임 rating 행렬을 만듭니다. 이 행렬로만 유사도를 계산해야
holdout(정답)이 모델에 새어 들어가지 않습니다.

In [5]:
users = np.sort(inter['steamid'].unique())
items = np.sort(inter['appid'].unique())
uidx = {u: i for i, u in enumerate(users)}
iidx = {it: i for i, it in enumerate(items)}
n_u, n_i = len(users), len(items)

R = np.zeros((n_u, n_i), dtype=np.float32)
for row in train_df.itertuples(index=False):
    R[uidx[row.steamid], iidx[row.appid]] = row.rating

owned_mask = (R > 0).astype(np.float32)
density = owned_mask.sum() / (n_u * n_i)
print(f"user-item 행렬: {n_u} x {n_i}, 밀도(density)={density:.4%}")

user-item 행렬: 779 x 3267, 밀도(density)=0.8239%


## 5. User-based CF

유저-유저 코사인 유사도로 이웃을 구하고, 이웃들의 rating을 유사도로 가중평균해 예측합니다.

$$\hat{r}_{u,i} = \frac{\sum_{v} sim(u,v) \cdot r_{v,i}}{\sum_{v: r_{v,i}>0} |sim(u,v)|}$$

이미 train에서 갖고 있던 게임은 추천 후보에서 제외합니다.

In [6]:
user_sim = cosine_similarity(R)
np.fill_diagonal(user_sim, 0)

num_u = user_sim @ R
den_u = user_sim @ owned_mask
pred_user_cf = np.divide(num_u, den_u, out=np.zeros_like(num_u), where=den_u > 0)
pred_user_cf[R > 0] = -np.inf  # 이미 보유한 게임 제외

print("User-based CF 예측 완료:", pred_user_cf.shape)

User-based CF 예측 완료: (779, 3267)


## 6. Item-based CF

아이템-아이템 코사인 유사도(같은 유저에게 함께 선택된 정도)를 구하고, 유저가 이미 플레이한
게임들과 유사한 게임에 높은 점수를 줍니다.

$$\hat{r}_{u,i} = \frac{\sum_{j} sim(i,j) \cdot r_{u,j}}{\sum_{j: r_{u,j}>0} |sim(i,j)|}$$

In [7]:
item_sim = cosine_similarity(R.T)
np.fill_diagonal(item_sim, 0)

num_i = R @ item_sim
den_i = owned_mask @ item_sim
pred_item_cf = np.divide(num_i, den_i, out=np.zeros_like(num_i), where=den_i > 0)
pred_item_cf[R > 0] = -np.inf

print("Item-based CF 예측 완료:", pred_item_cf.shape)

Item-based CF 예측 완료: (779, 3267)


## 7. 평가 — NDCG@10 / Precision@10 / Recall@10

논문 2장 3절의 NDCG 정의를 그대로 사용합니다. holdout으로 뺀 게임을 relevance=1로 두고,
예측 점수로 정렬한 Top-10 안에 얼마나 잘 들어오는지를 측정합니다.

$$DCG@K = \sum_{k=1}^{K} \frac{rel_k}{\log_2(k+1)}, \quad NDCG@K = \frac{DCG@K}{IDCG@K}$$

In [8]:
def evaluate(pred, K=10):
    ho_by_user = holdout_df.groupby('steamid')['appid'].apply(set)
    ndcgs, precs, recs = [], [], []
    for u in users:
        true_items = ho_by_user.get(u, set())
        if not true_items:
            continue
        true_idx = {iidx[a] for a in true_items if a in iidx}

        scores = pred[uidx[u]]
        top_idx = np.argpartition(-scores, K)[:K]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        rel = np.array([1.0 if idx in true_idx else 0.0 for idx in top_idx])
        dcg = np.sum(rel / np.log2(np.arange(2, K + 2)))
        n_rel = min(len(true_idx), K)
        idcg = np.sum(1.0 / np.log2(np.arange(2, n_rel + 2))) if n_rel > 0 else 0.0

        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
        precs.append(rel.sum() / K)
        recs.append(rel.sum() / len(true_idx))
    return np.mean(ndcgs), np.mean(precs), np.mean(recs)


ndcg_u, prec_u, rec_u = evaluate(pred_user_cf)
ndcg_i, prec_i, rec_i = evaluate(pred_item_cf)

result_df = pd.DataFrame({
    'model': ['UserCF', 'ItemCF'],
    'NDCG@10': [ndcg_u, ndcg_i],
    'Precision@10': [prec_u, prec_i],
    'Recall@10': [rec_u, rec_i],
})
result_df.to_csv(RESULTS_DIR / 'cf_evaluation_results.csv', index=False)
result_df

,model,NDCG@10,Precision@10,Recall@10
0,UserCF,0.000233,0.000128,0.000428
1,ItemCF,0.001205,0.000257,0.000856


## 8. 결과 해석 — 왜 점수가 낮은가

논문의 유저-게임 SVD 베이스라인은 NDCG@10 = 0.3227이었지만, 우리 데이터에서 단순 이웃 기반
UserCF/ItemCF는 그보다 훨씬 낮게 나옵니다. 원인을 데이터로 직접 확인합니다.

- 논문 데이터: 유저 12,761명 / 게임 9,321개 / 유저당 평균 6.1개 → 밀도 0.065%지만 **유저 수가 훨씬 많아** 이웃을 찾기 쉬움
- 우리 데이터(정제 후): 유저 779명 / 게임 3,267개 / 유저당 30개(고정) → 밀도(0.82%)는 더 높지만
  **유저 수가 적어 롱테일 게임의 co-occurrence가 거의 없음**
- 후보 게임 풀이 3,267개로 커서, Top-10 안에 정답을 맞추는 게 구조적으로 어려움 (희소성 문제)

아래에서 게임별 보유 유저 수 분포를 직접 확인합니다.

In [9]:
item_pop_train = train_df['appid'].value_counts()
print(f"train에서 1명만 보유한 게임: {(item_pop_train == 1).sum():,} / {len(item_pop_train):,}개 "
      f"({(item_pop_train == 1).mean():.1%})")

ho_item_pop = holdout_df['appid'].map(item_pop_train).fillna(0)
print(f"holdout 게임 중 train에 co-occurrence가 전혀 없는(원천적으로 복원 불가능한) 게임: "
      f"{(ho_item_pop == 0).sum():,} / {len(ho_item_pop):,}개 ({(ho_item_pop == 0).mean():.1%})")

print()
print("→ 대부분의 holdout 게임은 이론적으로 복원 가능(co-occurrence 존재)하지만, 후보 풀이 커서")
print("  단순 코사인 유사도 이웃 기반 CF로는 Top-10 안에 올리기 어려움을 확인.")
print("  개선 방향: (1) SVD/행렬분해 기반 모델(논문 방식)로 전환, (2) 후보 풀을 인기 게임 top-N으로")
print("  좁히기, (3) 최종모델.md의 태그/장르 기반 컨텐츠 벡터와 결합한 하이브리드 추천")

train에서 1명만 보유한 게임: 1,541 / 3,124개 (49.3%)
holdout 게임 중 train에 co-occurrence가 전혀 없는(원천적으로 복원 불가능한) 게임: 144 / 2,337개 (6.2%)

→ 대부분의 holdout 게임은 이론적으로 복원 가능(co-occurrence 존재)하지만, 후보 풀이 커서
  단순 코사인 유사도 이웃 기반 CF로는 Top-10 안에 올리기 어려움을 확인.
  개선 방향: (1) SVD/행렬분해 기반 모델(논문 방식)로 전환, (2) 후보 풀을 인기 게임 top-N으로
  좁히기, (3) 최종모델.md의 태그/장르 기반 컨텐츠 벡터와 결합한 하이브리드 추천


## 9. 정성적 확인 — 유저별 추천 게임 샘플

수치만으로는 감이 안 오니, 실제로 몇 명의 유저에게 어떤 게임이 추천되는지 확인합니다.

In [10]:
def top_n_recommendations(pred, steamid, n=10):
    ui = uidx[steamid]
    scores = pred[ui]
    top_idx = np.argsort(-scores)[:n]
    return pd.DataFrame({
        'appid': items[top_idx],
        'game_name': [game_names.get(a, 'Unknown') for a in items[top_idx]],
        'score': scores[top_idx],
    })


sample_user = users[0]
print(f"샘플 유저: {sample_user}")
print("\n실제 보유 게임 중 검증용으로 뺀(맞춰야 하는) 게임:")
print(holdout_df[holdout_df['steamid'] == sample_user][['appid']].merge(
    game_names.rename('game_name'), left_on='appid', right_index=True))

print("\nUserCF Top-10 추천:")
display(top_n_recommendations(pred_user_cf, sample_user))

print("\nItemCF Top-10 추천:")
display(top_n_recommendations(pred_item_cf, sample_user))

샘플 유저: 76561197960612825

실제 보유 게임 중 검증용으로 뺀(맞춰야 하는) 게임:
       appid                 game_name
213   435150  Divinity: Original Sin 2
214   892970                   Valheim
215  1173220     Bleak Faith: Forsaken

UserCF Top-10 추천:


,appid,game_name,score
0,347620,Gaokao.Love.100Days,1.000000
1,368340,CrossCode,1.000000
2,2426960,Summoners War,1.000000
3,1668940,崩坏3,1.000000
4,382310,Eco,1.000000
5,1891700,Tap Ninja,1.000000
6,836620,Black Desert (Retired),1.000000
7,3629260,EA SPORTS FC™ 26 SHOWCASE,0.997262
8,344760,Reign Of Kings,0.942261
9,1399720,Antimatter Dimensions,0.884411



ItemCF Top-10 추천:


,appid,game_name,score
0,357190,Ultimate Marvel vs. Capcom 3,1.0
1,470310,TROUBLESHOOTER: Abandoned Children,1.0
2,1135810,Vault of the Void,1.0
3,1836730,Echo Point Nova,1.0
4,1616110,Glitch Busters,1.0
5,3595270,Call of Duty®: Modern Warfare® III,1.0
6,2310,Quake,1.0
7,2349820,Hero's Land,1.0
8,1181830,Urtuk: The Desolation,1.0
9,274940,Depth,1.0


## 10. SVD 기반 CF (행렬분해, Matrix Factorization)

UserCF/ItemCF는 이웃 기반(memory-based)이라 co-occurrence가 적으면 예측이 거의 불가능했습니다.
논문 2장 1절의 SVD 공식을 그대로 구현해 비교합니다 (편향 항 포함, SGD로 학습).

$$\hat{r}_{u,i} = \mu + b_u + b_i + p_u \cdot q_i$$

$$\underset{p,q,b}{\arg\min} \sum_{(u,i) \in train} (r_{u,i} - \hat{r}_{u,i})^2 + \lambda(\|p_u\|^2 + \|q_i\|^2 + b_u^2 + b_i^2)$$

`scikit-surprise` 패키지(논문에서 사용)가 이 환경엔 없어서, 동일한 수식을 numpy로 직접 구현합니다.
논문이 GridSearchCV로 `n_factors`, `lr_all`, `reg_all`을 튜닝한 것처럼, 여기서도 몇 가지 조합을
holdout NDCG@10 기준으로 간단히 비교해 최적 설정을 고릅니다.

In [11]:
def train_svd(train_df, n_u, n_i, uidx, iidx, n_factors, n_epochs, lr, reg, seed=SEED):
    svd_rng = np.random.default_rng(seed)
    mu = train_df['rating'].mean()
    bu = np.zeros(n_u)
    bi = np.zeros(n_i)
    P = svd_rng.normal(0, 0.1, (n_u, n_factors))
    Q = svd_rng.normal(0, 0.1, (n_i, n_factors))

    U = train_df['steamid'].map(uidx).values
    I = train_df['appid'].map(iidx).values
    Rr = train_df['rating'].values
    order = np.arange(len(Rr))

    for epoch in range(n_epochs):
        svd_rng.shuffle(order)
        for idx in order:
            u, i, r = U[idx], I[idx], Rr[idx]
            pred = mu + bu[u] + bi[i] + P[u] @ Q[i]
            e = r - pred
            bu[u] += lr * (e - reg * bu[u])
            bi[i] += lr * (e - reg * bi[i])
            p_old = P[u].copy()
            P[u] += lr * (e * Q[i] - reg * P[u])
            Q[i] += lr * (e * p_old - reg * Q[i])
    return mu, bu, bi, P, Q


def svd_predict(mu, bu, bi, P, Q, owned_mask):
    pred = mu + bu[:, None] + bi[None, :] + P @ Q.T
    pred[owned_mask] = -np.inf
    return pred


owned = R > 0
svd_configs = [
    dict(n_factors=10, n_epochs=20, lr=0.01, reg=0.05),
    dict(n_factors=20, n_epochs=20, lr=0.01, reg=0.05),
    dict(n_factors=30, n_epochs=20, lr=0.005, reg=0.05),
]

svd_search = []
for cfg in svd_configs:
    mu, bu, bi, P, Q = train_svd(train_df, n_u, n_i, uidx, iidx, **cfg)
    pred = svd_predict(mu, bu, bi, P, Q, owned)
    ndcg, prec, rec = evaluate(pred)
    svd_search.append({**cfg, 'NDCG@10': ndcg})

svd_search_df = pd.DataFrame(svd_search)
svd_search_df

,n_factors,n_epochs,lr,reg,NDCG@10
0,10,20,0.010,0.05,0.030178
1,20,20,0.010,0.05,0.030464
2,30,20,0.005,0.05,0.034411


In [12]:
best_cfg = svd_search_df.loc[svd_search_df['NDCG@10'].idxmax(), ['n_factors', 'n_epochs', 'lr', 'reg']].to_dict()
best_cfg = {k: (int(v) if k in ('n_factors', 'n_epochs') else float(v)) for k, v in best_cfg.items()}
print('최적 설정:', best_cfg)

mu, bu, bi, P, Q = train_svd(train_df, n_u, n_i, uidx, iidx, **best_cfg)
pred_svd = svd_predict(mu, bu, bi, P, Q, owned)

ndcg_svd, prec_svd, rec_svd = evaluate(pred_svd)
print(f'SVD  NDCG@10={ndcg_svd:.4f} Precision@10={prec_svd:.4f} Recall@10={rec_svd:.4f}')

최적 설정: {'n_factors': 30, 'n_epochs': 20, 'lr': 0.005, 'reg': 0.05}


SVD  NDCG@10=0.0344 Precision@10=0.0128 Recall@10=0.0428


## 11. 컨텐츠 기반 유저 취향 벡터 (최종모델.md ②③ 단계)

`docs/최종모델.md`가 설계한 대로, 게임의 장르/태그 원핫 벡터를 유저가 보유한 게임들의 이용시간으로
가중합해 **User Preference Vector**를 만들고, 게임 Feature Vector와 코사인 유사도를 구합니다.

⚠️ 팀이 만든 `user_genre/tag_preference_*_clean.csv`는 **유저의 30개 게임 전체**로 계산되어 있어
holdout으로 뺀 3개(정답)도 가중치에 포함되어 있습니다 — 그대로 쓰면 검증 데이터가 모델에 새어 들어가는
data leakage가 생깁니다. 따라서 여기서는 `train_df`(27개)만으로 **다시 계산**합니다.

**추가로, 유저 질문에 대한 A/B 비교**: 팀 파이프라인은 `log(playtime+1)` 가중치를 쓰는데,
"장르별 실제 이용시간을 그대로 쓰면 어떨까"라는 제안을 검증하기 위해 raw playtime 가중치 버전도
같이 계산해서 NDCG@10으로 비교합니다.

In [13]:
gf = pd.read_csv(DATA_DIR / 'game_features_full.csv')
content_cols = [c for c in gf.columns if c.startswith('genre__') or c.startswith('tag__')]
game_content_df = gf.set_index('appid')[content_cols].reindex(items).fillna(0.0)
game_content = game_content_df.values
print(f"게임 컨텐츠 벡터: {game_content.shape} (장르 12 + 태그 50)")

train_c = train_df.merge(game_content_df.reset_index(), on='appid', how='left')


def build_user_content_pref(weight_col):
    weighted = train_c[content_cols].values * train_c[weight_col].values[:, None]
    tmp = pd.DataFrame(weighted, columns=content_cols)
    tmp['steamid'] = train_c['steamid'].values
    pref = tmp.groupby('steamid').sum().reindex(users).fillna(0.0)
    row_sum = pref.sum(axis=1)
    return pref.div(row_sum.replace(0, np.nan), axis=0).fillna(0.0).values


train_c['log_playtime'] = np.log1p(train_c['playtime_hours'])
user_content_log = build_user_content_pref('log_playtime')
user_content_raw = build_user_content_pref('playtime_hours')

content_score_log = cosine_similarity(user_content_log, game_content)
content_score_raw = cosine_similarity(user_content_raw, game_content)
content_score_log[owned] = -np.inf
content_score_raw[owned] = -np.inf

ndcg_log, prec_log, rec_log = evaluate(content_score_log)
ndcg_raw, prec_raw, rec_raw = evaluate(content_score_raw)

pd.DataFrame({
    'weighting': ['log(playtime+1) [팀 기존 방식]', 'raw playtime_hours [제안]'],
    'NDCG@10': [ndcg_log, ndcg_raw],
    'Precision@10': [prec_log, prec_raw],
    'Recall@10': [rec_log, rec_raw],
})

게임 컨텐츠 벡터: (3267, 62) (장르 12 + 태그 50)


,weighting,NDCG@10,Precision@10,Recall@10
0,log(playtime+1) [팀 기존 방식],0.025526,0.010398,0.034660
1,raw playtime_hours [제안],0.021864,0.009114,0.030381


**결과**: `log(playtime+1)` 가중치가 raw playtime 가중치보다 NDCG@10이 더 높게 나옵니다.
이유는 이용시간 분포가 매우 skew되어 있어서(한 게임에 300시간+ 몰아넣은 유저가 흔함), raw playtime을
그대로 쓰면 **가장 많이 한 게임 1개의 장르/태그가 유저 취향 벡터 전체를 지배**해버리기 때문입니다.
`log`을 씌우면 그 격차가 완화되어 여러 게임의 장르/태그가 골고루 반영되고, 결과적으로 취향을
더 안정적으로 나타냅니다. → **팀의 기존 `log(playtime+1)` 방식을 그대로 유지**하고, 아래 하이브리드
모델에도 이 버전(`content_score_log`)을 사용합니다.

## 12. 하이브리드 (SVD + 컨텐츠) 추천

최종모델.md ⑤단계("추천 점수 = 유사도 + 다른 요소, 가중치는 실험으로 결정")대로, SVD 예측 점수와
컨텐츠 코사인 유사도를 정규화 후 가중합해 최종 추천 점수를 만듭니다. 가중치 α는 홀드아웃 NDCG@10이
가장 좋은 값을 실험으로 선택합니다.

$$score_{u,i} = \alpha \cdot \widetilde{svd}_{u,i} + (1-\alpha) \cdot \widetilde{content}_{u,i}$$

(물결표는 유저별 min-max 정규화를 의미 — 두 점수의 스케일이 달라 그대로 더하면 한쪽이 지배하게 됨)

In [14]:
def normalize_rows_masked(mat, owned_mask):
    m = np.where(~owned_mask, mat, np.nan)
    row_min = np.nanmin(m, axis=1, keepdims=True)
    row_max = np.nanmax(m, axis=1, keepdims=True)
    denom = row_max - row_min
    denom[denom == 0] = 1.0
    return (mat - row_min) / denom


svd_norm = normalize_rows_masked(pred_svd, owned)
content_norm = normalize_rows_masked(content_score_log, owned)

alpha_results = []
best_hybrid = None
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    hybrid = alpha * svd_norm + (1 - alpha) * content_norm
    hybrid[owned] = -np.inf
    ndcg, prec, rec = evaluate(hybrid)
    alpha_results.append({'alpha (SVD 비중)': alpha, 'NDCG@10': ndcg, 'Precision@10': prec, 'Recall@10': rec})
    if best_hybrid is None or ndcg > best_hybrid[1]:
        best_hybrid = (alpha, ndcg, hybrid)

alpha_df = pd.DataFrame(alpha_results)
display(alpha_df)

best_alpha = best_hybrid[0]
pred_hybrid = best_hybrid[2]
print(f"최적 alpha={best_alpha} (NDCG@10={best_hybrid[1]:.4f})")

,alpha (SVD 비중),NDCG@10,Precision@10,Recall@10
0,0.00,0.025526,0.010398,0.034660
1,0.25,0.033628,0.014121,0.047069
2,0.50,0.046376,0.020796,0.069320
3,0.75,0.040365,0.015533,0.051776
4,1.00,0.034411,0.012837,0.042790


최적 alpha=0.5 (NDCG@10=0.0464)


## 13. 전체 모델 비교

In [15]:
final_result_df = pd.DataFrame({
    'model': ['UserCF', 'ItemCF', 'SVD', 'Content-only', f'Hybrid (alpha={best_alpha})'],
    'NDCG@10': [
        *evaluate(pred_user_cf)[:1],
        *evaluate(pred_item_cf)[:1],
        ndcg_svd,
        ndcg_log,
        best_hybrid[1],
    ],
})
prec_rec = [evaluate(pred_user_cf), evaluate(pred_item_cf), (ndcg_svd, prec_svd, rec_svd),
            (ndcg_log, prec_log, rec_log), evaluate(pred_hybrid)]
final_result_df['Precision@10'] = [p for _, p, _ in prec_rec]
final_result_df['Recall@10'] = [r for _, _, r in prec_rec]

final_result_df.to_csv(RESULTS_DIR / 'cf_evaluation_results.csv', index=False)
final_result_df.sort_values('NDCG@10', ascending=False).reset_index(drop=True)

,model,NDCG@10,Precision@10,Recall@10
0,Hybrid (alpha=0.5),0.046376,0.020796,0.069320
1,SVD,0.034411,0.012837,0.042790
2,Content-only,0.025526,0.010398,0.034660
3,ItemCF,0.001205,0.000257,0.000856
4,UserCF,0.000233,0.000128,0.000428


## 14. 기준선(Baseline)과 비교 — 이 점수가 실제로 의미가 있나?

NDCG@10=0.04~0.05가 절대적으로 낮아 보이는 게 맞는지 확인하기 위해, 두 가지 기준선과 비교합니다.

- **Random**: 후보 게임에 무작위 점수 부여
- **Popularity**: 개인화 없이, train에서 가장 많은 유저가 보유한 게임 순으로 모든 유저에게 동일하게 추천
  (가장 단순한 "베스트셀러 추천")

In [16]:
# Random baseline (5회 평균)
rand_ndcgs = []
for trial in range(5):
    pred_rand = rng.random((n_u, n_i))
    pred_rand[owned] = -np.inf
    ndcg, _, _ = evaluate(pred_rand)
    rand_ndcgs.append(ndcg)
ndcg_rand = np.mean(rand_ndcgs)

# Popularity baseline: 모든 유저에게 동일한 점수(=train에서의 보유 유저 수)
pop_count = train_df['appid'].value_counts()
pop_score = np.zeros(n_i)
for a, c in pop_count.items():
    pop_score[iidx[a]] = c
pred_pop = np.tile(pop_score, (n_u, 1))
pred_pop[owned] = -np.inf
ndcg_pop, prec_pop, rec_pop = evaluate(pred_pop)

baseline_df = pd.DataFrame({
    'model': ['Random', 'Popularity(비개인화)', 'UserCF', 'ItemCF', 'Content-only', 'SVD', f'Hybrid (alpha={best_alpha})'],
    'NDCG@10': [ndcg_rand, ndcg_pop, *[evaluate(p)[0] for p in [pred_user_cf, pred_item_cf, content_score_log, pred_svd, pred_hybrid]]],
})
baseline_df = baseline_df.sort_values('NDCG@10', ascending=False).reset_index(drop=True)
baseline_df.to_csv(RESULTS_DIR / 'cf_evaluation_results.csv', index=False)
baseline_df

,model,NDCG@10
0,Popularity(비개인화),0.103122
1,Hybrid (alpha=0.5),0.046376
2,SVD,0.034411
3,Content-only,0.025526
4,Random,0.001264
5,ItemCF,0.001205
6,UserCF,0.000233


**결과 해석**: Random(≈0.002) 대비로는 우리 모델들이 확실히 신호가 있지만,
**Popularity 기준선(개인화 전혀 없이 "인기 게임 그대로 추천") 하나가 SVD·Content·Hybrid를 전부 이깁니다.**

즉, 지금까지 만든 "개인화 추천"이 사실상 **"모두에게 같은 인기 게임 리스트를 주는 것"보다도 개인화
효과를 못 내고 있다**는 뜻입니다. 원인은 유저 수(779명)가 너무 적어서, 특정 유저의 취향과 진짜
비슷한 이웃/latent factor를 찾을 만큼의 데이터가 없고, 반대로 "인기 게임"은 애초에 이 779명의
라이브러리 안에서 여러 명이 공통으로 갖고 있어서 나온 것이라 holdout 게임도 우연히 맞힐 확률이
구조적으로 높기 때문입니다 (인기 게임일수록 애초에 holdout으로 뽑힐 확률도 높음).

**결론적으로 지금 단계에서는 "정교한 모델을 더 잘 튜닝"하는 것보다 (1) 유저 수를 늘리거나
(2) Popularity를 하이브리드에 명시적으로 포함시키는 것이 NDCG@10을 올리는 데 훨씬 효과적일 가능성이 높습니다.**

## 15. Popularity를 포함한 3-way 하이브리드

14절에서 확인했듯 비개인화 Popularity 기준선(0.103)이 SVD·Content·기존 Hybrid를 전부 이겼습니다.
원인을 더 보기 위해 SVD의 epoch을 20→50→100→150으로 늘려봤는데, **NDCG가 오히려 계속 떨어졌습니다**
(0.034 → 0.028 → 0.017 → 0.009). 데이터가 작다 보니 오래 학습할수록 latent factor가 과적합되면서
아이템 편향 `b_i`가 담고 있던 popularity 신호까지 같이 희석되는 것으로 보입니다. 즉 "SVD를 더 잘
학습시키기"는 해법이 아니었습니다.

대신 popularity를 무시하지 않고 **하이브리드의 세 번째 항으로 명시적으로 포함**시킵니다.

$$score_{u,i} = \alpha \cdot \widetilde{svd}_{u,i} + \beta \cdot \widetilde{content}_{u,i} + \gamma \cdot \widetilde{pop}_i, \quad \alpha+\beta+\gamma=1$$

가중치는 12절과 동일하게 holdout NDCG@10 기준 그리드서치로 정합니다.

In [17]:
epoch_sweep = []
for n_epochs in [20, 50, 100, 150]:
    mu_e, bu_e, bi_e, P_e, Q_e = train_svd(train_df, n_u, n_i, uidx, iidx,
                                            n_factors=30, n_epochs=n_epochs, lr=0.005, reg=0.05)
    pred_e = svd_predict(mu_e, bu_e, bi_e, P_e, Q_e, owned)
    ndcg_e, _, _ = evaluate(pred_e)
    epoch_sweep.append({'n_epochs': n_epochs, 'NDCG@10': ndcg_e})

pd.DataFrame(epoch_sweep)

,n_epochs,NDCG@10
0,20,0.034411
1,50,0.027776
2,100,0.016509
3,150,0.008595


In [18]:
pop_count = train_df['appid'].value_counts()
pop_score = np.zeros(n_i)
for a, c in pop_count.items():
    pop_score[iidx[a]] = c
pred_pop = np.tile(pop_score, (n_u, 1))
pop_norm = normalize_rows_masked(pred_pop, owned)

pred_pop_masked = pred_pop.copy()
pred_pop_masked[owned] = -np.inf
ndcg_pop, prec_pop, rec_pop = evaluate(pred_pop_masked)
print(f'Popularity(비개인화)  NDCG@10={ndcg_pop:.4f} Precision@10={prec_pop:.4f} Recall@10={rec_pop:.4f}')

grid3_results = []
for a in [0.0, 0.05, 0.1, 0.15, 0.2]:
    for b in [0.2, 0.3, 0.4, 0.5, 0.6]:
        g = round(1.0 - a - b, 3)
        if g < 0:
            continue
        combo = a * svd_norm + b * content_norm + g * pop_norm
        combo[owned] = -np.inf
        ndcg, prec, rec = evaluate(combo)
        grid3_results.append({'svd_w': a, 'content_w': b, 'pop_w': g, 'NDCG@10': ndcg, 'Precision@10': prec, 'Recall@10': rec})

grid3_df = pd.DataFrame(grid3_results).sort_values('NDCG@10', ascending=False).reset_index(drop=True)
grid3_df.head(10)

Popularity(비개인화)  NDCG@10=0.1031 Precision@10=0.0401 Recall@10=0.1335


,svd_w,content_w,pop_w,NDCG@10,Precision@10,Recall@10
0,0.05,0.4,0.55,0.115071,0.045058,0.150193
1,0.00,0.5,0.50,0.114845,0.044801,0.149337
2,0.10,0.4,0.50,0.114607,0.044416,0.148053
3,0.15,0.4,0.45,0.113647,0.043389,0.144630
4,0.00,0.4,0.60,0.113607,0.044416,0.148053
5,0.05,0.5,0.45,0.113510,0.043517,0.145058
6,0.10,0.5,0.40,0.112369,0.042875,0.142918
7,0.20,0.3,0.50,0.112004,0.043517,0.145058
8,0.10,0.3,0.60,0.111882,0.043774,0.145914
9,0.15,0.3,0.55,0.111522,0.043517,0.145058


In [19]:
best3 = grid3_df.iloc[0]
w_svd, w_content, w_pop = best3['svd_w'], best3['content_w'], best3['pop_w']

pred_hybrid3 = w_svd * svd_norm + w_content * content_norm + w_pop * pop_norm
pred_hybrid3[owned] = -np.inf
ndcg_h3, prec_h3, rec_h3 = evaluate(pred_hybrid3)

print(f"최적 조합: SVD={w_svd}, Content={w_content}, Popularity={w_pop}")
print(f"3-way Hybrid  NDCG@10={ndcg_h3:.4f} Precision@10={prec_h3:.4f} Recall@10={rec_h3:.4f}")
print(f"Popularity 대비 개선: {(ndcg_h3/ndcg_pop - 1)*100:+.1f}%")

최적 조합: SVD=0.05, Content=0.4, Popularity=0.55
3-way Hybrid  NDCG@10=0.1151 Precision@10=0.0451 Recall@10=0.1502
Popularity 대비 개선: +11.6%


**결과**: Content와 Popularity를 5:5~4:6 비율로 섞고 SVD를 소량(0~0.1)만 더한 조합이 가장 좋고,
Popularity 단독(0.103) 대비 **NDCG@10이 약 +11~12% 개선**됩니다 — popularity를 이기는 방향으로 실제로
움직였습니다. SVD 비중이 커질수록 오히려 성능이 떨어지는 경향은 그리드서치 표에서도 일관되게
보입니다 (데이터가 작아 SVD의 latent factor가 아직 노이즈에 가깝기 때문으로 해석됩니다).

## 16. 전체 모델 비교 (최종)

In [20]:
all_results = {
    'UserCF': evaluate(pred_user_cf),
    'ItemCF': evaluate(pred_item_cf),
    'SVD': (ndcg_svd, prec_svd, rec_svd),
    'Content-only': (ndcg_log, prec_log, rec_log),
    f'Hybrid (SVD+Content, alpha={best_alpha})': evaluate(pred_hybrid),
    'Popularity (비개인화)': (ndcg_pop, prec_pop, rec_pop),
    f'Hybrid (SVD+Content+Pop, {w_svd}/{w_content}/{w_pop})': (ndcg_h3, prec_h3, rec_h3),
    'Random': (ndcg_rand, np.nan, np.nan),
}

final_all_df = pd.DataFrame([
    {'model': k, 'NDCG@10': v[0], 'Precision@10': v[1], 'Recall@10': v[2]}
    for k, v in all_results.items()
]).sort_values('NDCG@10', ascending=False).reset_index(drop=True)

final_all_df.to_csv(RESULTS_DIR / 'cf_evaluation_results.csv', index=False)
final_all_df

,model,NDCG@10,Precision@10,Recall@10
0,"Hybrid (SVD+Content+Pop, 0.05/0.4/0.55)",0.115071,0.045058,0.150193
1,Popularity (비개인화),0.103122,0.040051,0.133504
2,"Hybrid (SVD+Content, alpha=0.5)",0.046376,0.020796,0.069320
3,SVD,0.034411,0.012837,0.042790
4,Content-only,0.025526,0.010398,0.034660
5,Random,0.001264,NaN,NaN
6,ItemCF,0.001205,0.000257,0.000856
7,UserCF,0.000233,0.000128,0.000428


## 17. 1명만 보유한 롱테일 게임을 제외하고 재실행

8절에서 확인했듯 train 게임의 절반 가까이(49.3%)가 유저 1명만 보유하고 있어서, 이런 게임은
UserCF/ItemCF가 애초에 이웃을 찾을 수 없는 원인이었습니다. 이번엔 **전체 데이터셋에서 1명만
보유한 게임(appid)을 통째로 제외**하고 처음부터 다시 (rating 생성 → train/holdout 분리 →
모든 모델) 돌려서, 후보 풀을 줄이는 게 실제로 도움이 되는지 확인합니다.

먼저 필터링이 유저별 보유 게임 수를 얼마나 줄이는지 확인합니다 (너무 많이 줄면 홀드아웃 분리 자체가
불가능해질 수 있음).

In [21]:
item_pop_all = inter['appid'].value_counts()
n_singleton = (item_pop_all == 1).sum()
keep_items_f = item_pop_all[item_pop_all >= 2].index

inter_f = inter[inter['appid'].isin(keep_items_f)].reset_index(drop=True)
games_per_user_f = inter_f.groupby('steamid').size()

print(f"제외된 게임(1명만 보유): {n_singleton:,} / {inter['appid'].nunique():,}개")
print(f"필터링 후 게임 수: {inter_f['appid'].nunique():,}개")
print(f"필터링 후 유저별 보유 게임 수: min={games_per_user_f.min()}, mean={games_per_user_f.mean():.1f}, max={games_per_user_f.max()}")

제외된 게임(1명만 보유): 1,586 / 3,267개
필터링 후 게임 수: 1,681개
필터링 후 유저별 보유 게임 수: min=10, mean=28.0, max=30


필터링 후에도 유저당 최소 10개는 남아 홀드아웃 10% 분리에 문제없습니다. 이제 rating 생성부터
전체 파이프라인을 다시 돌립니다 (동일 `SEED=42`, 동일 로직 — `_f` 접미사로 구분).

In [22]:
user_max_f = inter_f.groupby('steamid')['playtime_hours'].transform('max')
inter_f = inter_f.copy()
inter_f['rating'] = np.where(user_max_f > 0, inter_f['playtime_hours'] / user_max_f, 0.0)

rng_f = np.random.default_rng(SEED)
is_holdout_f = np.zeros(len(inter_f), dtype=bool)
for uid, g in inter_f.groupby('steamid'):
    n = len(g)
    k = max(1, round(n * HOLDOUT_FRAC))
    chosen = rng_f.choice(g.index.values, size=k, replace=False)
    is_holdout_f[chosen] = True

train_df_f = inter_f[~is_holdout_f].reset_index(drop=True)
holdout_df_f = inter_f[is_holdout_f].reset_index(drop=True)

users_f = np.sort(inter_f['steamid'].unique())
items_f = np.sort(inter_f['appid'].unique())
uidx_f = {u: i for i, u in enumerate(users_f)}
iidx_f = {it: i for i, it in enumerate(items_f)}
n_u_f, n_i_f = len(users_f), len(items_f)

R_f = np.zeros((n_u_f, n_i_f), dtype=np.float32)
for row in train_df_f.itertuples(index=False):
    R_f[uidx_f[row.steamid], iidx_f[row.appid]] = row.rating
owned_f = R_f > 0

density_f = owned_f.sum() / (n_u_f * n_i_f)
print(f"필터링 후 행렬: {n_u_f} x {n_i_f}, 밀도={density_f:.4%} (필터링 전: {density:.4%})")


def evaluate_f(pred, K=10):
    ho_by_user = holdout_df_f.groupby('steamid')['appid'].apply(set)
    ndcgs, precs, recs = [], [], []
    for u in users_f:
        true_items = ho_by_user.get(u, set())
        if not true_items:
            continue
        true_idx = {iidx_f[a] for a in true_items if a in iidx_f}
        scores = pred[uidx_f[u]]
        top_idx = np.argpartition(-scores, K)[:K]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        rel = np.array([1.0 if idx in true_idx else 0.0 for idx in top_idx])
        dcg = np.sum(rel / np.log2(np.arange(2, K + 2)))
        n_rel = min(len(true_idx), K)
        idcg = np.sum(1.0 / np.log2(np.arange(2, n_rel + 2))) if n_rel > 0 else 0.0
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
        precs.append(rel.sum() / K)
        recs.append(rel.sum() / len(true_idx))
    return np.mean(ndcgs), np.mean(precs), np.mean(recs)

필터링 후 행렬: 779 x 1681, 밀도=1.4878% (필터링 전: 0.8239%)


In [23]:
# UserCF / ItemCF
user_sim_f = cosine_similarity(R_f); np.fill_diagonal(user_sim_f, 0)
num_u_f = user_sim_f @ R_f
den_u_f = user_sim_f @ owned_f.astype(np.float32)
pred_user_f = np.divide(num_u_f, den_u_f, out=np.zeros_like(num_u_f), where=den_u_f > 0)
pred_user_f[owned_f] = -np.inf

item_sim_f = cosine_similarity(R_f.T); np.fill_diagonal(item_sim_f, 0)
num_i_f = R_f @ item_sim_f
den_i_f = owned_f.astype(np.float32) @ item_sim_f
pred_item_f = np.divide(num_i_f, den_i_f, out=np.zeros_like(num_i_f), where=den_i_f > 0)
pred_item_f[owned_f] = -np.inf

# SVD (동일 최적 설정 재사용)
mu_f, bu_f, bi_f, P_f, Q_f = train_svd(train_df_f, n_u_f, n_i_f, uidx_f, iidx_f, **best_cfg)
pred_svd_f = svd_predict(mu_f, bu_f, bi_f, P_f, Q_f, owned_f)

# Content (train만 사용, log(playtime+1) 가중)
game_content_df_f = gf.set_index('appid')[content_cols].reindex(items_f).fillna(0.0)
game_content_f = game_content_df_f.values
train_c_f = train_df_f.merge(game_content_df_f.reset_index(), on='appid', how='left')
train_c_f['log_playtime'] = np.log1p(train_c_f['playtime_hours'])
weighted_f = train_c_f[content_cols].values * train_c_f['log_playtime'].values[:, None]
tmp_f = pd.DataFrame(weighted_f, columns=content_cols)
tmp_f['steamid'] = train_c_f['steamid'].values
pref_f = tmp_f.groupby('steamid').sum().reindex(users_f).fillna(0.0)
user_content_f = pref_f.div(pref_f.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0).values
content_score_f = cosine_similarity(user_content_f, game_content_f)
pred_content_f = content_score_f.copy(); pred_content_f[owned_f] = -np.inf

# Popularity
pop_count_f = train_df_f['appid'].value_counts()
pop_score_f = np.zeros(n_i_f)
for a, c in pop_count_f.items():
    pop_score_f[iidx_f[a]] = c
pred_pop_f = np.tile(pop_score_f, (n_u_f, 1))
pred_pop_f_masked = pred_pop_f.copy(); pred_pop_f_masked[owned_f] = -np.inf

print('UserCF ', evaluate_f(pred_user_f)[0])
print('ItemCF ', evaluate_f(pred_item_f)[0])
print('SVD    ', evaluate_f(pred_svd_f)[0])
print('Content', evaluate_f(pred_content_f)[0])
print('Pop    ', evaluate_f(pred_pop_f_masked)[0])

UserCF  0.000774898722811801
ItemCF  0.0019454409208880747
SVD     0.0357180688236431
Content 0.0329149401616614
Pop     0.10390586378623204


In [24]:
svd_norm_f = normalize_rows_masked(pred_svd_f, owned_f)
content_norm_f = normalize_rows_masked(content_score_f, owned_f)
pop_norm_f = normalize_rows_masked(pred_pop_f, owned_f)

grid_f_results = []
for a in [0.0, 0.05, 0.1, 0.15, 0.2]:
    for b in [0.2, 0.3, 0.4, 0.5, 0.6]:
        g = round(1.0 - a - b, 3)
        if g < 0:
            continue
        combo = a * svd_norm_f + b * content_norm_f + g * pop_norm_f
        combo[owned_f] = -np.inf
        ndcg, prec, rec = evaluate_f(combo)
        grid_f_results.append({'svd_w': a, 'content_w': b, 'pop_w': g, 'NDCG@10': ndcg, 'Precision@10': prec, 'Recall@10': rec})

grid_f_df = pd.DataFrame(grid_f_results).sort_values('NDCG@10', ascending=False).reset_index(drop=True)
best_f = grid_f_df.iloc[0]
pred_hybrid_f = best_f['svd_w'] * svd_norm_f + best_f['content_w'] * content_norm_f + best_f['pop_w'] * pop_norm_f
pred_hybrid_f[owned_f] = -np.inf
ndcg_hf, prec_hf, rec_hf = evaluate_f(pred_hybrid_f)

grid_f_df.head(5)

,svd_w,content_w,pop_w,NDCG@10,Precision@10,Recall@10
0,0.05,0.4,0.55,0.111248,0.042490,0.146128
1,0.15,0.4,0.45,0.111062,0.042490,0.146128
2,0.10,0.4,0.50,0.110977,0.042490,0.146341
3,0.00,0.5,0.50,0.110307,0.041463,0.142918
4,0.00,0.4,0.60,0.110126,0.041849,0.143774


In [25]:
compare_df = pd.DataFrame({
    'model': ['UserCF', 'ItemCF', 'SVD', 'Content-only', 'Popularity', '3-way Hybrid (best)'],
    'NDCG@10 (필터 전, 3267개 게임)': [
        evaluate(pred_user_cf)[0], evaluate(pred_item_cf)[0], ndcg_svd, ndcg_log, ndcg_pop, ndcg_h3,
    ],
    'NDCG@10 (필터 후, 1681개 게임)': [
        evaluate_f(pred_user_f)[0], evaluate_f(pred_item_f)[0], evaluate_f(pred_svd_f)[0],
        evaluate_f(pred_content_f)[0], evaluate_f(pred_pop_f_masked)[0], ndcg_hf,
    ],
})
compare_df['개선율'] = (compare_df['NDCG@10 (필터 후, 1681개 게임)'] / compare_df['NDCG@10 (필터 전, 3267개 게임)'] - 1)
compare_df

,model,"NDCG@10 (필터 전, 3267개 게임)","NDCG@10 (필터 후, 1681개 게임)",개선율
0,UserCF,0.000233,0.000775,2.325108
1,ItemCF,0.001205,0.001945,0.614710
2,SVD,0.034411,0.035718,0.037983
3,Content-only,0.025526,0.032915,0.289454
4,Popularity,0.103122,0.103906,0.007601
5,3-way Hybrid (best),0.115071,0.111248,-0.033220


In [26]:
weak_models = {
    'UserCF': (pred_user_cf, pred_user_f),
    'ItemCF': (pred_item_cf, pred_item_f),
    'Content-only': (content_score_log, pred_content_f),
}

weak_rows = []
for name, (pred_before, pred_after) in weak_models.items():
    ndcg_b, prec_b, rec_b = evaluate(pred_before)
    ndcg_a, prec_a, rec_a = evaluate_f(pred_after)
    weak_rows.append({'model': name, 'stage': '필터 전 (3267개)', 'NDCG@10': ndcg_b, 'Precision@10': prec_b, 'Recall@10': rec_b})
    weak_rows.append({'model': name, 'stage': '필터 후 (1681개)', 'NDCG@10': ndcg_a, 'Precision@10': prec_a, 'Recall@10': rec_a})

weak_compare_df = pd.DataFrame(weak_rows)
weak_compare_df.to_csv(RESULTS_DIR / 'cf_longtail_filter_weak_models.csv', index=False)
weak_compare_df

,model,stage,NDCG@10,Precision@10,Recall@10
0,UserCF,필터 전 (3267개),0.000233,0.000128,0.000428
1,UserCF,필터 후 (1681개),0.000775,0.000385,0.001712
2,ItemCF,필터 전 (3267개),0.001205,0.000257,0.000856
3,ItemCF,필터 후 (1681개),0.001945,0.000642,0.002353
4,Content-only,필터 전 (3267개),0.025526,0.010398,0.034660
5,Content-only,필터 후 (1681개),0.032915,0.013350,0.046641


**결과 해석**: 밀도는 0.82%→1.49%로 거의 2배 올랐고, 그 덕에 **UserCF(+233%), ItemCF(+61%),
Content(+29%)는 상대적으로 크게 개선**됐습니다 — 이웃/co-occurrence 기반 방법일수록 롱테일 제거의
덕을 많이 봅니다.

하지만 정작 지금까지 가장 성능이 좋았던 **3-way Hybrid는 오히려 살짝 하락**했습니다(0.1151→0.1112).
이유는 최고 성능 모델이 이미 popularity 비중을 55~60%나 차지하고 있었는데, popularity 신호 자체는
게임 풀을 줄인다고 더 좋아지지 않기 때문입니다. 즉, **롱테일 게임 제외는 "약한 개별 모델"을 개선하는
데는 유효하지만, 이미 popularity가 주도하는 최고 성능 하이브리드에는 추가 이득이 없었습니다.**

정리하면, 두 접근(롱테일 제외 vs popularity 결합)은 서로 대체재라기보다 **어느 모델을 쓰느냐에 따라
다른 해법**이라는 게 이번 실험의 결론입니다.

## 18. tag30 벡터 기반 UserCF / ItemCF / (SVD 참고)

지금까지 UserCF/ItemCF는 **게임ID 자체**(3,267차원, 49%가 1인 보유라 희소)를 기준으로 유사도를
계산했습니다. 이번엔 `user_tag_preference_top30_normalized_clean.csv`의 **태그30 취향 벡터**(30차원,
모든 유저·게임에 항상 값이 있어 조밀함)를 유사도 계산 기준으로 바꿔서 다시 구합니다.

- **UserCF**: 유저-유저 유사도를 게임 rating 벡터 대신 태그30 취향 벡터로 계산 (예측 자체는 여전히
  실제 `train` rating의 가중평균)
- **ItemCF**: 게임-게임 유사도를 co-occurrence 대신 각 게임의 태그30 원핫 벡터로 계산
- **Content-only**: 기존 genre(12)+tag50(50)=62차원 대신 tag30(30차원)만 사용해 재계산
- **SVD**: 태그를 전혀 쓰지 않는 모델이라 태그30을 적용할 지점이 없습니다. 비교 기준으로 기존 결과만
  다시 표에 포함합니다.

In [27]:
t30 = pd.read_csv(DATA_DIR / 'user_tag_preference_top30_normalized_clean.csv')
tag30_cols = [c for c in t30.columns if c.startswith('tag__')]
print(f"태그30 컬럼 수: {len(tag30_cols)}")

game_tag30_df = gf.set_index('appid')[tag30_cols].reindex(items).fillna(0.0)
game_tag30 = game_tag30_df.values

train_t30 = train_df.merge(game_tag30_df.reset_index(), on='appid', how='left')
train_t30['log_playtime'] = np.log1p(train_t30['playtime_hours'])
weighted_t30 = train_t30[tag30_cols].values * train_t30['log_playtime'].values[:, None]
tmp_t30 = pd.DataFrame(weighted_t30, columns=tag30_cols)
tmp_t30['steamid'] = train_t30['steamid'].values
pref_t30 = tmp_t30.groupby('steamid').sum().reindex(users).fillna(0.0)
user_tag30 = pref_t30.div(pref_t30.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0).values

print(f"게임 태그30 벡터: {game_tag30.shape}, 유저 태그30 취향 벡터: {user_tag30.shape}")

태그30 컬럼 수: 30
게임 태그30 벡터: (3267, 30), 유저 태그30 취향 벡터: (779, 30)


In [28]:
# UserCF: 유저 유사도를 tag30 취향 벡터로 계산
user_sim_tag30 = cosine_similarity(user_tag30)
np.fill_diagonal(user_sim_tag30, 0)
print(f"tag30 유저 유사도 평균={user_sim_tag30.mean():.3f} (게임ID 기준 UserCF는 유저 대부분 유사도가 0에 가까웠음)")

num_u_t30 = user_sim_tag30 @ R
den_u_t30 = user_sim_tag30 @ owned.astype(np.float32)
pred_user_tag30 = np.divide(num_u_t30, den_u_t30, out=np.zeros_like(num_u_t30), where=den_u_t30 > 0)
pred_user_tag30[owned] = -np.inf

# ItemCF: 게임 유사도를 tag30 원핫 벡터로 계산
item_sim_tag30 = cosine_similarity(game_tag30)
np.fill_diagonal(item_sim_tag30, 0)
num_i_t30 = R @ item_sim_tag30
den_i_t30 = owned.astype(np.float32) @ item_sim_tag30
pred_item_tag30 = np.divide(num_i_t30, den_i_t30, out=np.zeros_like(num_i_t30), where=den_i_t30 > 0)
pred_item_tag30[owned] = -np.inf

# Content-only: tag30만 사용
content_tag30 = cosine_similarity(user_tag30, game_tag30)
pred_content_tag30 = content_tag30.copy()
pred_content_tag30[owned] = -np.inf

ndcg_u_t30, prec_u_t30, rec_u_t30 = evaluate(pred_user_tag30)
ndcg_i_t30, prec_i_t30, rec_i_t30 = evaluate(pred_item_tag30)
ndcg_c_t30, prec_c_t30, rec_c_t30 = evaluate(pred_content_tag30)

tag30_result_df = pd.DataFrame({
    'model': ['UserCF (게임ID 유사도)', 'UserCF (tag30 유사도)',
              'ItemCF (co-occurrence 유사도)', 'ItemCF (tag30 유사도)',
              'Content (genre+tag50)', 'Content (tag30만)',
              'SVD (참고, 태그 미사용)'],
    'NDCG@10': [evaluate(pred_user_cf)[0], ndcg_u_t30,
                evaluate(pred_item_cf)[0], ndcg_i_t30,
                ndcg_log, ndcg_c_t30,
                ndcg_svd],
    'Precision@10': [evaluate(pred_user_cf)[1], prec_u_t30,
                      evaluate(pred_item_cf)[1], prec_i_t30,
                      prec_log, prec_c_t30,
                      prec_svd],
    'Recall@10': [evaluate(pred_user_cf)[2], rec_u_t30,
                   evaluate(pred_item_cf)[2], rec_i_t30,
                   rec_log, rec_c_t30,
                   rec_svd],
})
tag30_result_df.to_csv(RESULTS_DIR / 'cf_tag30_comparison.csv', index=False)
tag30_result_df

tag30 유저 유사도 평균=0.774 (게임ID 기준 UserCF는 유저 대부분 유사도가 0에 가까웠음)


,model,NDCG@10,Precision@10,Recall@10
0,UserCF (게임ID 유사도),0.000233,0.000128,0.000428
1,UserCF (tag30 유사도),0.000000,0.000000,0.000000
2,ItemCF (co-occurrence 유사도),0.001205,0.000257,0.000856
3,ItemCF (tag30 유사도),0.000434,0.000257,0.000856
4,Content (genre+tag50),0.025526,0.010398,0.034660
5,Content (tag30만),0.019363,0.008216,0.027386
6,"SVD (참고, 태그 미사용)",0.034411,0.012837,0.042790


**결과**: 셋 다 tag30 기준으로 바꾸니 오히려 더 나빠졌습니다. 특히 UserCF(tag30)는 NDCG@10=0에
가깝게 완전히 무너졌는데, 원인은 위 셀에 출력된 **유저 간 평균 유사도가 0.77로 매우 높다**는
점입니다. 태그가 30개뿐이라 다들 "Singleplayer", "Great Soundtrack"처럼 흔한 태그 위주로 취향이
겹쳐 보여서, 사실상 거의 모든 유저가 "이웃"이 되어버립니다. 그러면 예측이 개인차가 사라진
평균적인 인기 취향으로 수렴해서, 홀드아웃으로 뺀 개인 취향 게임(니치한 조합)을 오히려 더 못
찾아냅니다.

즉 게임ID 기반 유사도는 **너무 희소(specific)해서** 이웃을 못 찾았던 반면, tag30 기반 유사도는
**너무 뭉뚱그려서(coarse)** 구분을 못 하는 정반대 실패입니다. Content(genre+tag50, 62차원)가 그
중간 지점이라 tag30(30차원)보다 나았던 것도 같은 이유 — 차원이 더 많을수록 유저/게임을 세밀하게
구분할 수 있기 때문입니다.

## 19. 개별 게임 중 playtime_hours=0인 row 제거 후 재실행

`clearingdataset` PR이 이미 **30개 게임 전부 0시간인 유저 99명**을 제거했지만(1절), 그 기준을 통과한
779명 중에도 "이 게임은 갖고 있지만 한 번도 안 켠" 개별 게임(row)이 남아있을 수 있습니다. 이번엔
그런 **row 단위 0시간 게임**까지 전부 제거하고 rating 생성부터 다시 돌립니다.

In [29]:
before_rows = len(inter)
inter_nz = inter[inter['playtime_hours'] > 0].copy().reset_index(drop=True)
print(f"0시간 row 제거: {before_rows:,} -> {len(inter_nz):,}개 ({before_rows - len(inter_nz)}개 제거)")

affected_users = inter[inter['playtime_hours'] == 0]['steamid'].nunique()
print(f"영향받은 유저 수: {affected_users} / {inter['steamid'].nunique()}")

gpu_nz = inter_nz.groupby('steamid').size()
print(f"제거 후 유저별 게임 수: min={gpu_nz.min()}, mean={gpu_nz.mean():.1f}, max={gpu_nz.max()}")

0시간 row 제거: 23,370 -> 23,297개 (73개 제거)
영향받은 유저 수: 9 / 779
제거 후 유저별 게임 수: min=8, mean=29.9, max=30


In [30]:
user_max_nz = inter_nz.groupby('steamid')['playtime_hours'].transform('max')
inter_nz['rating'] = np.where(user_max_nz > 0, inter_nz['playtime_hours'] / user_max_nz, 0.0)

rng_nz = np.random.default_rng(SEED)
is_holdout_nz = np.zeros(len(inter_nz), dtype=bool)
for uid, g in inter_nz.groupby('steamid'):
    n = len(g)
    k = max(1, round(n * HOLDOUT_FRAC))
    chosen = rng_nz.choice(g.index.values, size=k, replace=False)
    is_holdout_nz[chosen] = True

train_df_nz = inter_nz[~is_holdout_nz].reset_index(drop=True)
holdout_df_nz = inter_nz[is_holdout_nz].reset_index(drop=True)

users_nz = np.sort(inter_nz['steamid'].unique())
items_nz = np.sort(inter_nz['appid'].unique())
uidx_nz = {u: i for i, u in enumerate(users_nz)}
iidx_nz = {it: i for i, it in enumerate(items_nz)}
n_u_nz, n_i_nz = len(users_nz), len(items_nz)

R_nz = np.zeros((n_u_nz, n_i_nz), dtype=np.float32)
for row in train_df_nz.itertuples(index=False):
    R_nz[uidx_nz[row.steamid], iidx_nz[row.appid]] = row.rating
owned_nz = R_nz > 0
print(f"행렬: {n_u_nz} x {n_i_nz}, 밀도={owned_nz.sum() / (n_u_nz * n_i_nz):.4%}")


def evaluate_nz(pred, K=10):
    ho_by_user = holdout_df_nz.groupby('steamid')['appid'].apply(set)
    ndcgs, precs, recs = [], [], []
    for u in users_nz:
        true_items = ho_by_user.get(u, set())
        if not true_items:
            continue
        true_idx = {iidx_nz[a] for a in true_items if a in iidx_nz}
        scores = pred[uidx_nz[u]]
        top_idx = np.argpartition(-scores, K)[:K]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        rel = np.array([1.0 if idx in true_idx else 0.0 for idx in top_idx])
        dcg = np.sum(rel / np.log2(np.arange(2, K + 2)))
        n_rel = min(len(true_idx), K)
        idcg = np.sum(1.0 / np.log2(np.arange(2, n_rel + 2))) if n_rel > 0 else 0.0
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
        precs.append(rel.sum() / K)
        recs.append(rel.sum() / len(true_idx))
    return np.mean(ndcgs), np.mean(precs), np.mean(recs)

행렬: 779 x 3256, 밀도=0.8266%


In [31]:
user_sim_nz = cosine_similarity(R_nz); np.fill_diagonal(user_sim_nz, 0)
num_u_nz = user_sim_nz @ R_nz
den_u_nz = user_sim_nz @ owned_nz.astype(np.float32)
pred_user_nz = np.divide(num_u_nz, den_u_nz, out=np.zeros_like(num_u_nz), where=den_u_nz > 0)
pred_user_nz[owned_nz] = -np.inf

item_sim_nz = cosine_similarity(R_nz.T); np.fill_diagonal(item_sim_nz, 0)
num_i_nz = R_nz @ item_sim_nz
den_i_nz = owned_nz.astype(np.float32) @ item_sim_nz
pred_item_nz = np.divide(num_i_nz, den_i_nz, out=np.zeros_like(num_i_nz), where=den_i_nz > 0)
pred_item_nz[owned_nz] = -np.inf

mu_nz, bu_nz, bi_nz, P_nz, Q_nz = train_svd(train_df_nz, n_u_nz, n_i_nz, uidx_nz, iidx_nz, **best_cfg)
pred_svd_nz = svd_predict(mu_nz, bu_nz, bi_nz, P_nz, Q_nz, owned_nz)

game_content_df_nz = gf.set_index('appid')[content_cols].reindex(items_nz).fillna(0.0)
game_content_nz = game_content_df_nz.values
train_c_nz = train_df_nz.merge(game_content_df_nz.reset_index(), on='appid', how='left')
train_c_nz['log_playtime'] = np.log1p(train_c_nz['playtime_hours'])
weighted_nz = train_c_nz[content_cols].values * train_c_nz['log_playtime'].values[:, None]
tmp_nz = pd.DataFrame(weighted_nz, columns=content_cols)
tmp_nz['steamid'] = train_c_nz['steamid'].values
pref_nz = tmp_nz.groupby('steamid').sum().reindex(users_nz).fillna(0.0)
user_content_nz = pref_nz.div(pref_nz.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0).values
content_score_nz = cosine_similarity(user_content_nz, game_content_nz)
pred_content_nz = content_score_nz.copy(); pred_content_nz[owned_nz] = -np.inf

pop_count_nz = train_df_nz['appid'].value_counts()
pop_score_nz = np.zeros(n_i_nz)
for a, c in pop_count_nz.items():
    pop_score_nz[iidx_nz[a]] = c
pred_pop_nz = np.tile(pop_score_nz, (n_u_nz, 1))
pred_pop_nz_masked = pred_pop_nz.copy(); pred_pop_nz_masked[owned_nz] = -np.inf

svd_norm_nz = normalize_rows_masked(pred_svd_nz, owned_nz)
content_norm_nz = normalize_rows_masked(content_score_nz, owned_nz)
pop_norm_nz = normalize_rows_masked(pred_pop_nz, owned_nz)

grid_nz_results = []
for a in [0.0, 0.05, 0.1, 0.15, 0.2]:
    for b in [0.2, 0.3, 0.4, 0.5, 0.6]:
        g = round(1.0 - a - b, 3)
        if g < 0:
            continue
        combo = a * svd_norm_nz + b * content_norm_nz + g * pop_norm_nz
        combo[owned_nz] = -np.inf
        ndcg, prec, rec = evaluate_nz(combo)
        grid_nz_results.append({'svd_w': a, 'content_w': b, 'pop_w': g, 'NDCG@10': ndcg, 'Precision@10': prec, 'Recall@10': rec})

grid_nz_df = pd.DataFrame(grid_nz_results).sort_values('NDCG@10', ascending=False).reset_index(drop=True)
best_nz = grid_nz_df.iloc[0]
pred_hybrid_nz = best_nz['svd_w'] * svd_norm_nz + best_nz['content_w'] * content_norm_nz + best_nz['pop_w'] * pop_norm_nz
pred_hybrid_nz[owned_nz] = -np.inf
ndcg_hnz, prec_hnz, rec_hnz = evaluate_nz(pred_hybrid_nz)

nz_compare_df = pd.DataFrame({
    'model': ['UserCF', 'ItemCF', 'SVD', 'Content-only', 'Popularity', '3-way Hybrid (best)'],
    'NDCG@10 (0시간 row 포함, 기존)': [
        evaluate(pred_user_cf)[0], evaluate(pred_item_cf)[0], ndcg_svd, ndcg_log, ndcg_pop, ndcg_h3,
    ],
    'NDCG@10 (0시간 row 제거)': [
        evaluate_nz(pred_user_nz)[0], evaluate_nz(pred_item_nz)[0], evaluate_nz(pred_svd_nz)[0],
        evaluate_nz(pred_content_nz)[0], evaluate_nz(pred_pop_nz_masked)[0], ndcg_hnz,
    ],
})
nz_compare_df.to_csv(RESULTS_DIR / 'cf_zero_playtime_row_filter.csv', index=False)
nz_compare_df

,model,"NDCG@10 (0시간 row 포함, 기존)",NDCG@10 (0시간 row 제거)
0,UserCF,0.000233,0.000000
1,ItemCF,0.001205,0.000777
2,SVD,0.034411,0.038345
3,Content-only,0.025526,0.021004
4,Popularity,0.103122,0.105135
5,3-way Hybrid (best),0.115071,0.116052


**결과**: 거의 차이가 없습니다. 제거된 row가 23,370개 중 73개(0.3%), 영향받은 유저도 9명뿐이라
(그중 최악의 경우도 게임 30개 중 8개만 남는 정도) 전체 지표에 유의미한 변화를 주지 못했습니다.
이는 **`clearingdataset` PR이 이미 핵심 문제(라이브러리 전체가 0시간인 유저 99명)를 제거했기 때문**으로
해석됩니다 — 남은 779명은 "가끔 안 하는 게임 한두 개"는 있어도 "완전히 신호 없는 유저"는 이미
없는 상태였던 것입니다. 따라서 이 필터는 데이터 품질상 해두는 게 맞지만, 지금 성능 병목(popularity
쏠림 + 유저 수 부족)을 해결해주는 방향은 아닙니다.

## 요약

| 산출물 | 경로 |
| --- | --- |
| Train interaction (90%) | `data/processed/cf_train_interactions.csv` |
| Holdout/검증 interaction (10%) | `data/processed/cf_holdout_interactions.csv` |
| 전체 모델 평가 결과 (8개 모델) | `results/cf_evaluation_results.csv` |

- 유저는 팀의 `clearingdataset` 정제 기준(`user_profile_features_filtered.csv`)을 따라
  **30개 게임 전부 `playtime_hours`가 0인 99명을 제외**하고 779명만 사용
- Rating은 유저별 `playtime_hours`를 0~1로 min-max 정규화해 사용 (논문 4장 2절 방식)
- 검증 데이터는 유저별 30개 게임 중 10%(3개)를 무작위 마스킹해 생성 (`SEED=42`)
- UserCF/ItemCF: 코사인 유사도 기반 이웃 가중평균 → 거의 무력 (NDCG@10 ≈ 0.0002~0.0012)
- SVD: 편향 포함 행렬분해(논문 2장 1절 수식), SGD 학습 → epoch을 늘릴수록 오히려 악화 (과적합)
- Content-only: 장르+태그 원핫과 유저 취향 벡터(log(playtime+1) 가중, train만 사용)의 코사인 유사도
- **Popularity(비개인화) 기준선(0.103)이 SVD·Content·초기 Hybrid를 전부 이김** — 유저 수(779명)가
  적어 개인화 신호보다 "다들 이 게임을 갖고 있다"는 신호가 더 강력했기 때문
- **SVD+Content+Popularity 3-way 하이브리드**로 popularity를 명시적으로 섞자 Popularity 단독보다
  **약 +11~12% NDCG@10 개선**에 성공 (최고 성능 모델)
- **결론**: 이 데이터 규모(779명)에서는 정교한 개인화 모델 단독보다, popularity를 기본 축으로 삼고
  content 기반 개인화를 소량 더하는 하이브리드가 가장 안정적으로 잘 작동함. 근본적으로는 유저 수가
  늘어나야 SVD/CF 계열의 개인화 신호가 popularity를 확실히 앞서기 시작할 것으로 예상됨.

**추가 실험(17절)**: 1명만 보유한 롱테일 게임(전체의 49%)을 제외하고 재실행한 결과, 밀도는
0.82%→1.49%로 올랐고 UserCF/ItemCF/Content는 크게 개선됐지만(+30~230%), 이미 popularity가 주도하는
3-way Hybrid 최고 성능 모델에는 추가 이득이 없었음 (오히려 -3%). 롱테일 제외는 이웃 기반 모델에는
유효한 전략이지만, popularity 결합 하이브리드의 대체재는 아님.

**추가 실험(18절)**: UserCF/ItemCF의 유사도 계산 기준을 게임ID(희소) 대신 tag30 취향/특성 벡터
(조밀, 30차원)로 바꿔봤으나 셋 다(UserCF·ItemCF·Content) 오히려 성능이 하락. 특히 UserCF(tag30)는
거의 0으로 붕괴 — 태그가 30개뿐이라 유저 간 평균 유사도가 0.77까지 치솟아 "이웃"의 개인차가
사라지고 평균적인 취향으로 수렴하기 때문. 게임ID 기반은 너무 희소해서, tag30 기반은 너무 뭉뚱그려서
실패하는 정반대 문제이며, genre+tag50(62차원)이 그 중간 지점이라 더 나은 성능을 보임.

**추가 실험(19절)**: 개별 game row 중 `playtime_hours=0`인 것(779명 중 9명 유저의 73개 row, 전체의
0.3%)까지 제거하고 재실행했으나 모든 모델에서 유의미한 변화 없음 — `clearingdataset` PR이 이미
핵심 문제(라이브러리 전체가 0시간인 유저)를 제거해 둔 상태였기 때문.